# MSAI635: Automating GPU Optimizations using RL
## End-to-End GRPO Training and Evaluation Pipeline
This notebook implements the reinforcement learning pipeline for Anna Thomas's project component. It covers environment configuration, teacher kernel verification, prompt construction, contrastive GRPO policy training, and greedy decoding evaluation on Google Colab Pro.

### 1. Hardware and Environment Verification
Verify CUDA GPU availability and configure the PyTorch memory allocator to mitigate fragmentation.

In [ ]:
import os

# Configure memory allocator to reduce fragmentation on A100 or RTX 3090 GPUs
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!nvidia-smi


### 2. Dependency Installation
Install the exact pinned software stack required for TRL GRPO, PEFT, bitsandbytes, and compilation tools.

In [ ]:
# Pinned dependencies from project specification
!pip install -q transformers==4.57.6 peft==0.17.1 bitsandbytes==0.48.1 \
    trl==0.23.1 ninja pybind11 accelerate datasets scipy

# Install local package in editable mode
!pip install -q -e .


### 3. Toolchain and Inline Compilation Sanity Check
Test that ninja, nvcc, and PyTorch load_inline can successfully compile a simple C++ and CUDA extension.

In [ ]:
import os
import shutil
from pathlib import Path

import torch
from torch.utils.cpp_extension import load_inline

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Device: {torch.cuda.get_device_name(0)}")

# Clear any stale cached build directory from previously failed attempts
cache_dir = Path.home() / ".cache" / "torch_extensions"
for stale in cache_dir.glob("*/sanity_check_mod*"):
    shutil.rmtree(stale, ignore_errors=True)

# Quick inline compilation sanity test
# Uses C-style casts (const float*) and quote includes to prevent browser HTML tag stripping
test_cuda = """
#include "torch/extension.h"
#include "cuda_runtime.h"

__global__ void add_kernel(const float* x, float* out, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) out[idx] = x[idx] + 1.0f;
}

torch::Tensor run_cuda(torch::Tensor x) {
    auto out = torch::empty_like(x);
    int n = x.numel();
    const float* in_ptr = (const float*)x.data_ptr();
    float* out_ptr = (float*)out.data_ptr();
    add_kernel<<<(n + 255) / 256, 256>>>(in_ptr, out_ptr, n);
    return out;
}
"""

cpp_src = """
#include "torch/extension.h"
torch::Tensor run_cuda(torch::Tensor x);
PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("run_cuda", &run_cuda, "Add kernel");
}
"""

mod = load_inline(
    name="sanity_check_mod",
    cpp_sources=cpp_src,
    cuda_sources=test_cuda,
    extra_cuda_cflags=["-O3", "--use_fast_math"],
)
x = torch.zeros(16, device="cuda")
res = mod.run_cuda(x)
print("Inline compilation sanity check passed! Sample output:", res[:4].cpu().tolist())


### 4. Teacher Kernel Verification and Benchmarking Test
Compile and verify numerical accuracy and latency of the curated teacher kernels from the example store.

In [ ]:
from data.example_store import TEACHER_EXAMPLES
from engine.benchmark import measure_speedup
from engine.compiler import compile_kernel
from engine.verifier import verify_numerical_correctness

print("Verifying Curated Teacher Kernels:")
for task_id, teachers in TEACHER_EXAMPLES.items():
    for teacher in teachers:
        name = teacher["name"]
        expected_speedup = teacher["speedup"]
        code = teacher["code"]

        mod, c_msg = compile_kernel(code)
        if mod is None:
            print(f"[FAIL] {name} failed to compile: {c_msg}")
            continue

        correct, v_msg = verify_numerical_correctness(mod, task_id)
        if not correct:
            print(f"[FAIL] {name} numerical check failed: {v_msg}")
            continue

        measured_speedup, is_stable = measure_speedup(mod, task_id, device="cuda")
        res_str = f'Measured: {measured_speedup:.2f}x (Stable: {is_stable})'
        print(f'[PASS] {name:<25} | OK | {expected_speedup:.2f}x | {res_str}')


### 5. Inspect Prompt Construction (Reference-Only vs Contrastive)
Compare prompts generated for Arm A (Scored Context) and Arm B (Reference Only).

In [ ]:
from engine.prompt_builder import build_prompt

print("=== Arm B (Reference Only) Prompt Sample ===")
print(build_prompt("swish", mode="reference_only")[:350] + "...\n")

print("=== Arm A (Scored Context / Contrastive) Prompt Sample ===")
print(build_prompt("swish", mode="contrastive")[:600] + "...\n")


### 6. Model Loading and Trainable Parameter Verification
Load model using 4-bit NormalFloat (NF4) quantization and attach LoRA adapter. For quick sanity runs on Colab Pro, set use_dev_model=True (Qwen2.5-Coder-7B). For production runs, set use_dev_model=False to load Qwen3-Coder-30B-A3B-Instruct.

In [ ]:
import gc

import torch

from configs.grpo_config import ExperimentConfig
from models.loader import load_model_and_tokenizer

cfg = ExperimentConfig()
# Set to True for fast testing on Colab (Tesla T4 or A100), False for 30B MoE model
cfg.use_dev_model = True

# Verification check: test model loading and verify trainable parameters
model, tokenizer = load_model_and_tokenizer(config=cfg, is_trainable=True)
print("Model and tokenizer successfully initialized and verified.")

# Crucial memory hygiene: Free the verification model to release GPU VRAM
# so GRPO training runs with maximum headroom without duplicate allocations
del model
del tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("GPU memory cleared. Ready for GRPO training loop.")


### 7. Run GRPO Policy Optimization (Matched Baseline)
Execute Group Relative Policy Optimization on Arm A (Contrastive) and Arm B (Reference-Only).

In [ ]:
import gc

import torch

from train_grpo import run_grpo_training

# Run Arm A (Scored Context / Contrastive)
print("Starting Training: Arm A (Contrastive)")
arm_a_path = run_grpo_training("arm_a", cfg, dry_run=False)

# Clean VRAM cache between training runs
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Run Arm B (Reference Only)
print("Starting Training: Arm B (Reference Only)")
arm_b_path = run_grpo_training("arm_b", cfg, dry_run=False)


### 8. Greedy Evaluation Across Tasks
Evaluate trained adapters using greedy decoding (temperature=0.0) on both training and holdout tasks.

In [ ]:
import gc

import torch
from peft import PeftModel

from evaluate import evaluate_checkpoint
from models.loader import load_model_and_tokenizer

# Clean GPU cache before evaluation
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load fresh base model in evaluation mode
base_model, tokenizer = load_model_and_tokenizer(config=cfg, is_trainable=False)

print("Evaluating Arm A Checkpoint:")
eval_model_a = PeftModel.from_pretrained(base_model, arm_a_path)
results_arm_a = evaluate_checkpoint(
    model=eval_model_a,
    tokenizer=tokenizer,
    task_keys=cfg.eval_tasks,
    prompt_mode="contrastive",
)

print("Evaluating Arm B Checkpoint:")
eval_model_b = PeftModel.from_pretrained(base_model, arm_b_path)
results_arm_b = evaluate_checkpoint(
    model=eval_model_b,
    tokenizer=tokenizer,
    task_keys=cfg.eval_tasks,
    prompt_mode="reference_only",
)


### 9. Statistical Comparison: Contrastive vs Reference-Only
Compare correctness rates, speedup thresholds, and median throughput gains.

In [ ]:
import pandas as pd

comparison_data = {
    "Metric": [
        "Total Evaluated Tasks",
        "Correctness Rate (%)",
        "Fast >= 1.0x Rate (%)",
        "Fast >= 1.1x Rate (%)",
        "Median Speedup (x)",
    ],
    "Arm A (Scored Context)": [
        results_arm_a["total_tasks"],
        f"{results_arm_a['correctness_rate'] * 100:.1f}%",
        f"{results_arm_a['fast_1_0_rate'] * 100:.1f}%",
        f"{results_arm_a['fast_1_1_rate'] * 100:.1f}%",
        f"{results_arm_a['median_speedup']:.2f}x",
    ],
    "Arm B (Reference Only)": [
        results_arm_b["total_tasks"],
        f"{results_arm_b['correctness_rate'] * 100:.1f}%",
        f"{results_arm_b['fast_1_0_rate'] * 100:.1f}%",
        f"{results_arm_b['fast_1_1_rate'] * 100:.1f}%",
        f"{results_arm_b['median_speedup']:.2f}x",
    ],
}

df = pd.DataFrame(comparison_data)
display(df)
